# E-Commerce Multi-Agent System Test Notebook

This notebook tests all domain agents in the e-Commerce Brain system.

**Agents:**
- Supervisor Agent - Intent detection and routing
- Sales Agent - Revenue and order analysis
- Inventory Agent - Stockout detection
- Marketing Agent - Campaign performance analysis
- Support Agent - Ticket volume and sentiment analysis

**All agents are traced via Langfuse for observability.**

In [33]:
# Cell 1: Setup - Add project root to path and configure logging
import sys
import os
import logging
import importlib

# Enable nested event loops (required for async in Jupyter)
import nest_asyncio
nest_asyncio.apply()

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Force reload of key modules to pick up changes
# This ensures we get the latest code including the direct database access fix
modules_to_reload = [
    'backend.database.queries',
    'backend.utils.data_loader',
    'backend.agents.base_agent',
    'backend.agents.inventory.agent',
    'backend.agents.sales.agent',
    'backend.agents.marketing.agent',
    'backend.agents.support.agent',
    'backend.graph',
]
for mod_name in modules_to_reload:
    if mod_name in sys.modules:
        try:
            importlib.reload(sys.modules[mod_name])
            print(f"🔄 Reloaded: {mod_name}")
        except Exception as e:
            print(f"⚠️ Could not reload {mod_name}: {e}")

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print(f"\n✅ Project root added to path: {project_root}")
print(f"✅ nest_asyncio applied (async operations enabled)")
print(f"💡 Note: Agents will auto-detect Jupyter and use direct database access")

🔄 Reloaded: backend.database.queries
🔄 Reloaded: backend.utils.data_loader
🔄 Reloaded: backend.agents.base_agent
🔄 Reloaded: backend.agents.inventory.agent
🔄 Reloaded: backend.agents.sales.agent
🔄 Reloaded: backend.agents.marketing.agent
🔄 Reloaded: backend.agents.support.agent
🔄 Reloaded: backend.graph

✅ Project root added to path: c:\Users\DevavarapuSaiRuthvik\Desktop\E-Commerce_Brain\e_commerce_brain
✅ nest_asyncio applied (async operations enabled)
💡 Note: Agents will auto-detect Jupyter and use direct database access


In [34]:
# Cell 2: Import all agents and verify configuration
from backend.settings import Settings
from backend.agents.supervisor.agent import SupervisorAgent
from backend.agents.sales.agent import SalesAgent
from backend.agents.inventory.agent import InventoryAgent
from backend.agents.marketing.agent import MarketingAgent
from backend.agents.support.agent import SupportAgent
from backend.agents.base_agent import AgentContext

print("=" * 60)
print("AGENT IMPORTS AND CONFIGURATION")
print("=" * 60)
print(f"✅ SupervisorAgent imported")
print(f"✅ SalesAgent imported")
print(f"✅ InventoryAgent imported")
print(f"✅ MarketingAgent imported")
print(f"✅ SupportAgent imported")
print(f"\n🔧 Azure Endpoint: {Settings.AZURE_ENDPOINT}")
print(f"🔧 API Version: {Settings.API_VERSION}")
print(f"🔧 Langfuse Session: {Settings.LANGFUSE_SESSION_ID}")
print(f"🔧 Langfuse User: {Settings.LANGFUSE_USER_ID}")
print("=" * 60)

AGENT IMPORTS AND CONFIGURATION
✅ SupervisorAgent imported
✅ SalesAgent imported
✅ InventoryAgent imported
✅ MarketingAgent imported
✅ SupportAgent imported

🔧 Azure Endpoint: https://ai-proxy.lab.epam.com
🔧 API Version: 2024-02-01
🔧 Langfuse Session: e-Commerce Multi Agents
🔧 Langfuse User: 001


## Test 1: Supervisor Agent - Intent Detection

Test the Supervisor Agent's ability to detect user intent from a question.

In [35]:
# Cell 3: Test Supervisor Agent
print("🔄 Initializing Supervisor Agent...")
supervisor = SupervisorAgent()

# Test questions for different intents
test_questions = [
    "Why did sales drop yesterday?",
    "We have stockout issues with our products",
    "What's happening with our marketing campaigns?",
    "Too many customer complaints today"
]

print("\n📋 Testing Intent Detection:")
print("-" * 60)

for question in test_questions:
    print(f"\n❓ Question: {question}")
    intent = supervisor.detect_intent(question)
    print(f"✅ Detected Intent: {intent}")

🔄 Initializing Supervisor Agent...


2026-01-27 17:29:50,201 - backend.agents.supervisor.agent - INFO - [SupervisorAgent] Initialized with model: gpt-4 and Langfuse tracing
2026-01-27 17:29:50,201 - backend.agents.supervisor.agent - INFO - [SupervisorAgent] Calling LLM for intent detection...



📋 Testing Intent Detection:
------------------------------------------------------------

❓ Question: Why did sales drop yesterday?


2026-01-27 17:29:51,330 - backend.agents.supervisor.agent - INFO - [SupervisorAgent] Detected intent: sales_drop
2026-01-27 17:29:51,330 - backend.agents.supervisor.agent - INFO - [SupervisorAgent] Calling LLM for intent detection...


✅ Detected Intent: sales_drop

❓ Question: We have stockout issues with our products


...

✅ Detected Intent: inventory_issue

❓ Question: What's happening with our marketing campaigns?


2026-01-27 17:29:53,201 - backend.agents.supervisor.agent - INFO - [SupervisorAgent] Detected intent: marketing_issue
2026-01-27 17:29:53,207 - backend.agents.supervisor.agent - INFO - [SupervisorAgent] Calling LLM for intent detection...


✅ Detected Intent: marketing_issue

❓ Question: Too many customer complaints today


2026-01-27 17:29:53,858 - backend.agents.supervisor.agent - INFO - [SupervisorAgent] Detected intent: support_issue


✅ Detected Intent: support_issue


## Test 2: Supervisor Agent - Full State Processing

Test the Supervisor Agent's `__call__` method that processes the full state.

In [36]:
# Cell 4: Test Supervisor Full State Processing
print("🔄 Testing Supervisor Agent's full state processing...")

# Create initial state
initial_state = {
    "question": "Why did sales drop yesterday?",
    "intent": "",
    "agents_to_call": [],
    "agent_outputs": {},
    "agents_completed": []
}

# Process through supervisor
result_state = supervisor(initial_state)

print("\n📊 Result State:")
print("-" * 60)
print(f"✅ Intent: {result_state.get('intent')}")
print(f"✅ Agents to call: {result_state.get('agents_to_call')}")
print(f"✅ Error: {result_state.get('error', 'None')}")

2026-01-27 17:30:01,010 - backend.agents.supervisor.agent - INFO - [SupervisorAgent] Processing question: Why did sales drop yesterday?
2026-01-27 17:30:01,011 - backend.agents.supervisor.agent - INFO - [SupervisorAgent] Calling LLM for intent detection...


🔄 Testing Supervisor Agent's full state processing...


2026-01-27 17:30:02,511 - backend.agents.supervisor.agent - INFO - [SupervisorAgent] Detected intent: sales_drop
2026-01-27 17:30:02,512 - backend.agents.supervisor.router - INFO - [Router] Intent 'sales_drop' → Agents: ['sales', 'inventory', 'marketing', 'support']
2026-01-27 17:30:02,512 - backend.agents.supervisor.agent - INFO - [SupervisorAgent] Intent: sales_drop | Routing to agents: ['sales', 'inventory', 'marketing', 'support']



📊 Result State:
------------------------------------------------------------
✅ Intent: sales_drop
✅ Agents to call: ['sales', 'inventory', 'marketing', 'support']
✅ Error: None


## Test 3: Domain Agents Initialization

Initialize and verify all domain agents are working correctly.

**Note:** These agents require the MCP server to be running for data loading.

In [37]:
# Cell 5: Initialize all domain agents
print("🔄 Initializing Domain Agents...")
print("-" * 60)

try:
    sales_agent = SalesAgent()
    print("✅ SalesAgent initialized")
except Exception as e:
    print(f"❌ SalesAgent failed: {e}")

try:
    inventory_agent = InventoryAgent()
    print("✅ InventoryAgent initialized")
except Exception as e:
    print(f"❌ InventoryAgent failed: {e}")

try:
    marketing_agent = MarketingAgent()
    print("✅ MarketingAgent initialized")
except Exception as e:
    print(f"❌ MarketingAgent failed: {e}")

try:
    support_agent = SupportAgent()
    print("✅ SupportAgent initialized")
except Exception as e:
    print(f"❌ SupportAgent failed: {e}")

print("-" * 60)
print("✅ All agents initialized!")

2026-01-27 17:30:05,827 - backend.utils.data_loader - INFO - [DirectDataLoader] Initialized (direct database access)


🔄 Initializing Domain Agents...
------------------------------------------------------------


2026-01-27 17:30:06,408 - backend.utils.llm_formatter - INFO - [LLMFormatter:sales] Initialized with Langfuse tracing
2026-01-27 17:30:06,408 - backend.agents.base_agent - INFO - [salesAgent] Initialized (loader: direct)
2026-01-27 17:30:06,408 - backend.utils.data_loader - INFO - [DirectDataLoader] Initialized (direct database access)


✅ SalesAgent initialized


2026-01-27 17:30:07,006 - backend.utils.llm_formatter - INFO - [LLMFormatter:inventory] Initialized with Langfuse tracing
2026-01-27 17:30:07,006 - backend.agents.base_agent - INFO - [inventoryAgent] Initialized (loader: direct)
2026-01-27 17:30:07,006 - backend.utils.data_loader - INFO - [DirectDataLoader] Initialized (direct database access)


✅ InventoryAgent initialized


2026-01-27 17:30:07,602 - backend.utils.llm_formatter - INFO - [LLMFormatter:marketing] Initialized with Langfuse tracing
2026-01-27 17:30:07,604 - backend.agents.base_agent - INFO - [marketingAgent] Initialized (loader: direct)
2026-01-27 17:30:07,604 - backend.utils.data_loader - INFO - [DirectDataLoader] Initialized (direct database access)


✅ MarketingAgent initialized


2026-01-27 17:30:08,221 - backend.utils.llm_formatter - INFO - [LLMFormatter:support] Initialized with Langfuse tracing
2026-01-27 17:30:08,223 - backend.agents.base_agent - INFO - [supportAgent] Initialized (loader: direct)


✅ SupportAgent initialized
------------------------------------------------------------
✅ All agents initialized!


## Test 4: Run Complete Graph

Run the complete LangGraph workflow with a sample question.

**Prerequisites:**
- Database must be seeded (`python scripts/seed_postgres.py`)
- **No MCP Server needed!** The agents auto-detect Jupyter and use direct database access.

**⚠️ If you see "fileno" or "event loop" errors:**
1. Restart the kernel (Kernel → Restart)
2. Run all cells from the beginning

In [39]:
# Cell 6: Run Complete Graph
# Force reimport to get latest code
import importlib
import sys

# Reload modules to ensure we have the latest direct loader code
for mod in ['backend.utils.data_loader', 'backend.agents.base_agent', 'backend.graph']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from backend.graph import run_graph, get_agent_findings, get_highest_confidence_agent

print("🔄 Running complete LangGraph workflow...")
print("=" * 60)

question = "Why did sales drop yesterday?"
print(f"❓ Question: {question}\n")

try:
    # Run the graph (all traces go to Langfuse)
    # Agents will auto-detect Jupyter and use direct database access
    final_state = run_graph(question)
    
    print("\n📊 RESULTS:")
    print("-" * 60)
    print(f"Intent: {final_state.get('intent', 'N/A')}")
    print(f"Agents Called: {final_state.get('agents_to_call', [])}")
    print(f"Agents Completed: {final_state.get('agents_completed', [])}")
    
    # Get findings
    findings = get_agent_findings(final_state)
    print("\n📝 Agent Findings:")
    for agent, finding in findings.items():
        print(f"  [{agent}]: {finding[:100]}...")
    
    # Get highest confidence
    if final_state.get("agent_outputs"):
        best_agent, confidence = get_highest_confidence_agent(final_state)
        print(f"\n🏆 Highest Confidence: {best_agent} ({confidence:.2%})")
    
    print("\n✅ Graph execution complete!")
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()
    print("\n⚠️ If you see 'fileno' or 'event loop' errors:")
    print("   1. Restart the kernel (Kernel → Restart)")
    print("   2. Run all cells from the beginning")
    print("\n⚠️ Make sure database is seeded:")
    print("   python scripts/seed_postgres.py")

2026-01-27 17:30:55,378 - backend.graph - INFO - [Graph] Starting execution with question: 'Why did sales drop yesterday?'
2026-01-27 17:30:55,378 - backend.graph - INFO - [Graph] Building graph...


🔄 Running complete LangGraph workflow...
❓ Question: Why did sales drop yesterday?



2026-01-27 17:30:55,965 - backend.agents.supervisor.agent - INFO - [SupervisorAgent] Initialized with model: gpt-4 and Langfuse tracing
2026-01-27 17:30:55,967 - backend.utils.data_loader - INFO - [DirectDataLoader] Initialized (direct database access)
2026-01-27 17:30:56,545 - backend.utils.llm_formatter - INFO - [LLMFormatter:inventory] Initialized with Langfuse tracing
2026-01-27 17:30:56,554 - backend.agents.base_agent - INFO - [inventoryAgent] Initialized (loader: direct)
2026-01-27 17:30:56,555 - backend.utils.data_loader - INFO - [DirectDataLoader] Initialized (direct database access)
2026-01-27 17:30:57,141 - backend.utils.llm_formatter - INFO - [LLMFormatter:sales] Initialized with Langfuse tracing
2026-01-27 17:30:57,141 - backend.agents.base_agent - INFO - [salesAgent] Initialized (loader: direct)
2026-01-27 17:30:57,141 - backend.utils.data_loader - INFO - [DirectDataLoader] Initialized (direct database access)
2026-01-27 17:30:57,746 - backend.utils.llm_formatter - INFO - 


📊 RESULTS:
------------------------------------------------------------
Intent: sales_drop
Agents Called: ['sales', 'inventory', 'marketing', 'support']
Agents Completed: ['inventory', 'sales', 'marketing', 'support']

📝 Agent Findings:
  [inventory]: No stockouts were recorded on January 22, 2026, maintaining a consistent zero stockout rate, with no...
  [sales]: Revenue yesterday (₹87,171.89) was 16% higher than the daily average (₹74,950.66), despite a 3% drop...
  [marketing]: Campaign conversions dropped 6% yesterday (101 vs 107 avg) despite a 24% reduction in spend (₹38K vs...
  [support]: Support tickets increased 33% yesterday (8 vs 6 avg) with 100% negative sentiment, primarily driven ...

🏆 Highest Confidence: support (70.00%)

✅ Graph execution complete!


## Test 5: Test Individual Agent Execution (with Mock Data)

Test individual agents with mock data (no MCP server required).

In [40]:
# Cell 7: Test Sales Agent Logic (Mock Data)
from backend.agents.sales.logic import (
    calculate_revenue_drop,
    calculate_order_drop,
    calculate_aov_drop,
    analyze_drop_cause,
    calculate_confidence
)

print("🔄 Testing Sales Agent Logic with Mock Data...")
print("-" * 60)

# Mock sales data
mock_sales_data = {
    "yesterday_revenue": 8500.00,
    "avg_revenue": 10000.00,
    "yesterday_orders": 85,
    "avg_orders": 100,
    "yesterday_aov": 100.00,
    "avg_aov": 100.00
}

# Calculate metrics
revenue_drop = calculate_revenue_drop(mock_sales_data)
order_drop = calculate_order_drop(mock_sales_data)
aov_drop = calculate_aov_drop(mock_sales_data)
drop_cause = analyze_drop_cause(mock_sales_data)
confidence = calculate_confidence(revenue_drop, order_drop, aov_drop)

print(f"📉 Revenue Drop: {revenue_drop:.1f}%")
print(f"📉 Order Drop: {order_drop:.1f}%")
print(f"📉 AOV Drop: {aov_drop:.1f}%")
print(f"🔍 Drop Cause: {drop_cause}")
print(f"📊 Confidence: {confidence:.2%}")

2026-01-27 17:31:09,777 - backend.agents.sales.logic - INFO - [SalesLogic] Revenue drop: 15.00%
2026-01-27 17:31:09,778 - backend.agents.sales.logic - INFO - [SalesLogic] Confidence: 80.00%


🔄 Testing Sales Agent Logic with Mock Data...
------------------------------------------------------------
📉 Revenue Drop: 15.0%
📉 Order Drop: 15.0%
📉 AOV Drop: 0.0%
🔍 Drop Cause: order_count
📊 Confidence: 80.00%


In [41]:
# Cell 8: Test Inventory Agent Logic (Mock Data)
from backend.agents.inventory.logic import (
    calculate_stockout_severity,
    identify_critical_products,
    calculate_confidence as inv_calculate_confidence
)

print("🔄 Testing Inventory Agent Logic with Mock Data...")
print("-" * 60)

# Mock inventory data - using integer product IDs (as returned by MCP server)
mock_inventory_data = {
    "total_stockouts": 15,
    "stockout_products": [1, 2, 3, 7, 8, 12],  # Product IDs (1-5 are critical)
    "critical_products": [
        {"sku": "SKU-001", "product_id": 1, "revenue_impact": 5000},
        {"sku": "SKU-002", "product_id": 2, "revenue_impact": 3000}
    ]
}

mock_baseline = {
    "avg_daily_stockouts": 5.0
}

# Calculate metrics
severity = calculate_stockout_severity(mock_inventory_data, mock_baseline)
critical_products = identify_critical_products(mock_inventory_data)
confidence = inv_calculate_confidence(
    severity=severity,
    critical_count=len(critical_products),
    total_stockouts=mock_inventory_data["total_stockouts"]
)

print(f"📦 Stockout Severity: {severity:.1f}x baseline")
print(f"⚠️ Critical Products: {critical_products}")
print(f"📊 Confidence: {confidence:.2%}")

2026-01-27 17:31:12,857 - backend.agents.inventory.logic - INFO - [InventoryLogic] Stockout severity: 3.00x baseline
2026-01-27 17:31:12,858 - backend.agents.inventory.logic - INFO - [InventoryLogic] Critical products affected: 3
2026-01-27 17:31:12,859 - backend.agents.inventory.logic - INFO - [InventoryLogic] Confidence: 98.00%


🔄 Testing Inventory Agent Logic with Mock Data...
------------------------------------------------------------
📦 Stockout Severity: 3.0x baseline
⚠️ Critical Products: [1, 2, 3]
📊 Confidence: 98.00%


In [42]:
# Cell 9: Test Marketing Agent Logic (Mock Data)
from backend.agents.marketing.logic import (
    calculate_conversion_drop,
    calculate_spend_change,
    calculate_efficiency,
    analyze_campaign_status,
    calculate_confidence as mkt_calculate_confidence
)

print("🔄 Testing Marketing Agent Logic with Mock Data...")
print("-" * 60)

# Mock marketing data
mock_marketing_data = {
    "yesterday_conversions": 120,
    "avg_conversions": 200,
    "yesterday_spend": 5000.00,
    "avg_spend": 5500.00,
    "yesterday_active_campaigns": 5
}

# Calculate metrics
conversion_drop = calculate_conversion_drop(mock_marketing_data)
spend_change = calculate_spend_change(mock_marketing_data)
efficiency = calculate_efficiency(mock_marketing_data)
campaign_status = analyze_campaign_status(mock_marketing_data)
confidence = mkt_calculate_confidence(
    conversion_drop=conversion_drop,
    spend_change=spend_change,
    efficiency_drop=efficiency['efficiency_drop_pct']
)

print(f"📉 Conversion Drop: {conversion_drop:.1f}%")
print(f"💰 Spend Change: {spend_change:.1f}%")
print(f"📊 Efficiency Drop: {efficiency['efficiency_drop_pct']:.1f}%")
print(f"🎯 Active Campaigns: {campaign_status['active_campaigns']}")
print(f"📊 Confidence: {confidence:.2%}")

2026-01-27 17:31:16,966 - backend.agents.marketing.logic - INFO - [MarketingLogic] Conversion drop: 40.00%
2026-01-27 17:31:16,966 - backend.agents.marketing.logic - INFO - [MarketingLogic] Spend change: -9.09%
2026-01-27 17:31:16,967 - backend.agents.marketing.logic - INFO - [MarketingLogic] Confidence: 95.00%


🔄 Testing Marketing Agent Logic with Mock Data...
------------------------------------------------------------
📉 Conversion Drop: 40.0%
💰 Spend Change: -9.1%
📊 Efficiency Drop: 34.0%
🎯 Active Campaigns: 5
📊 Confidence: 95.00%


In [43]:
# Cell 10: Test Support Agent Logic (Mock Data)
from backend.agents.support.logic import (
    calculate_ticket_spike,
    analyze_sentiment,
    analyze_top_categories,
    calculate_confidence as sup_calculate_confidence
)

print("🔄 Testing Support Agent Logic with Mock Data...")
print("-" * 60)

# Mock support data
mock_support_data = {
    "yesterday_tickets": 150,
    "avg_tickets": 50,
    "yesterday_negative_pct": 65.0,
    "yesterday_negative_count": 98,
    "top_categories": [
        ("delivery_issues", 45),
        ("product_quality", 30),
        ("refund_request", 25)
    ]
}

# Calculate metrics
ticket_spike = calculate_ticket_spike(mock_support_data)
sentiment = analyze_sentiment(mock_support_data)
top_category, top_count, top_pct = analyze_top_categories(mock_support_data)
confidence = sup_calculate_confidence(
    spike_pct=ticket_spike,
    negative_pct=sentiment['negative_pct'],
    top_category_pct=top_pct
)

print(f"📈 Ticket Spike: {ticket_spike:.1f}%")
print(f"😠 Negative Sentiment: {sentiment['negative_pct']:.1f}% ({sentiment['negative_count']} tickets)")
print(f"🏷️ Top Category: {top_category} ({top_pct:.1f}%)")
print(f"📊 Confidence: {confidence:.2%}")

2026-01-27 17:31:17,470 - backend.agents.support.logic - INFO - [SupportLogic] Ticket spike: 200.00%
2026-01-27 17:31:17,471 - backend.agents.support.logic - INFO - [SupportLogic] Top category: delivery_issues (30.0%)
2026-01-27 17:31:17,472 - backend.agents.support.logic - INFO - [SupportLogic] Confidence: 93.00%


🔄 Testing Support Agent Logic with Mock Data...
------------------------------------------------------------
📈 Ticket Spike: 200.0%
😠 Negative Sentiment: 65.0% (98 tickets)
🏷️ Top Category: delivery_issues (30.0%)
📊 Confidence: 93.00%


## Test 6: Verify Langfuse Traces

Flush all traces to Langfuse and provide the dashboard link.

In [44]:
# Cell 11: Flush Langfuse Traces and Summary
from langfuse import Langfuse

# Initialize Langfuse client
langfuse_client = Langfuse(
    secret_key=Settings.LANGFUSE_SECRET_KEY,
    public_key=Settings.LANGFUSE_PUBLIC_KEY,
    host=Settings.LANGFUSE_BASE_URL
)

# Flush all pending traces
langfuse_client.flush()

print("=" * 60)
print("AGENT TESTS SUMMARY")
print("=" * 60)
print(f"""
✅ Supervisor Agent tested (intent detection + state processing)
✅ Sales Agent logic tested (revenue, orders, AOV analysis)
✅ Inventory Agent logic tested (stockout severity, critical products)
✅ Marketing Agent logic tested (conversion, spend, efficiency)
✅ Support Agent logic tested (ticket spike, sentiment, categories)
✅ All traces sent to Langfuse

📊 View traces at: {Settings.LANGFUSE_BASE_URL}

🔍 Filter by:
   - Session ID: {Settings.LANGFUSE_SESSION_ID}
   - User ID: {Settings.LANGFUSE_USER_ID}

📝 Expected trace names:
   - supervisor_detect_intent
   - supervisor_call
   - agent_execute
   - agent_call
   - run_graph
   - format_finding
   - data_loader_*
""")
print("=" * 60)

AGENT TESTS SUMMARY

✅ Supervisor Agent tested (intent detection + state processing)
✅ Sales Agent logic tested (revenue, orders, AOV analysis)
✅ Inventory Agent logic tested (stockout severity, critical products)
✅ Marketing Agent logic tested (conversion, spend, efficiency)
✅ Support Agent logic tested (ticket spike, sentiment, categories)
✅ All traces sent to Langfuse

📊 View traces at: https://cloud.langfuse.com

🔍 Filter by:
   - Session ID: e-Commerce Multi Agents
   - User ID: 001

📝 Expected trace names:
   - supervisor_detect_intent
   - supervisor_call
   - agent_execute
   - agent_call
   - run_graph
   - format_finding
   - data_loader_*

